# TTM × SIX — Functional Transfer: three-arm TTM-Small-4M exploratory pilot

**This notebook is PREPARED, not authorized to train.**  It does not load Qwen models or open original G1/TTM TEST. The Qwen features come only from the earlier owner-supplied 64-public-prompt ZIP. The separate G1 original scientific lock remains HOLD.

**Do not run the final training cell** until a NEW owner authorization specifically for `TTM_SIX_TTM4M_FUNCTIONAL_TRANSFER_PUBLIC48_DEV16_V1` has been issued. The code checks an external JSON approval receipt; this is a guard against accidental execution, not a cryptographic owner signature.

Input files needed: (1) the existing `TTM_SIX_G1_EXPLORE_REAL_QWEN_DEV_RESULTS.zip`, SHA-pinned; (2) frozen original TRAIN-only TTM BPE manifest `T1_BPE2048_REAL_TRAIN_MANIFEST.json`, SHA pinned; (3) original `step_1280` checkpoint folder on Drive with `READY.json` + `checkpoint.pt`, both SHA pinned. Input source code for P02/BPE is fetched at an immutable GitHub commit and its Git blob verified. **No original TinyStories, TRAIN/VAL pack, or private TEST is needed.**

Run **Runtime → Change runtime type → T4 GPU** if doing GPU preflight or later training. A free T4 is not guaranteed. Install the lightweight `regex` dependency only if missing. Save trained artifacts only under a brand-new `MyDrive/TTM/ttm-six-functional-pilot-v1/<run_id>` directory, never the historical TTM or G1 checkpoint path.


In [ ]:
# 1. Prepare exact local runner; does NOT load models, train, or mount Drive.
import base64, gzip, hashlib, os, pathlib, sys, subprocess
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
RUNNER_SHA256 = 'cdd1d5615368f2e4254c50af977887afa3969e4f6aabc6953b4c39d73dfc9cec'
_COMPRESSED_B64 = ('H4sIAAAAAAAC/7U8/XPiOLK/56/w3VU9mx3HAUKAMOOtYgiToTaBXCA7t8ejXP4QiS9gs7bJJjM1//vrbkm2bCCZvXu3dzXBstRqtVr9Lf/tLyfbNDnxwuiERU/a5iV7iKPTo7/+9a+z2bX2rE1H/+hpabxNfHb8xJJwGbLA1GbD6ex4mTCmZQ/w77GbrDXofzxdu6vVcetaY8+bVZy4WZy8aG7gbjI3C+PIOjoaM4CiJdtI+yPMHrQ4Ce/DyF3h6JPLhrZJwic3YzQBvCzep37IIp9pq9h/tLR+RDOEfphp4+GXo/iPCKBuwlWcae5mk8RPMGQZrpgWplrEfJamLmCyBIjZA9Mub+60jZs9WNrsATrA/91IRfmo1T1ptLWL4a/HcbR6gclhkZGpjSczDZD042gZJmvqamnjWMuY6z8AAus4YCvNBxqkiHwAaK1iN0gtpOfRMonXmuMst9k2YY6jhetNnAC+URRz8qRHsim537hJyuTzg5s+rEJPPvI/0GBts3CVt8by17/SOJK/45TPi8uFEXLSG3iUXRI3CuK1fEofVKDpS45UFq5zhL6GG6TukXyOtusNbDTQenN09Ddt+Oz6WbF3sOMpLs/UYKXaNmXJccI2cRhlrgdbtN6Kv0i9E0lLdxW6KQPSDf9xM7wdXQ/HM83WdOATB3jSgb+ta+fT3XgwG03G/StndtsfTz8Nb52bu49Xo0Gr68D2NdrOrw39aPJlDC/6Nze3k1/7VwiG/x46B8HdjK4mM2fWovG3d2NndIHjsmztpOGzA39ba9jKyMeNc1dOBlRMlyxxNlsP+BKmD9gTTP8Ew6eTu9vB0LkczZyPV5OPCMfvNpfLVt2vN7qddqfD2p2z08A79Zh31mYd//yUuZ2g4Z3mg6ef+82zNg49b3e7AfMaQaNxhj2XXqd+XmdnXsNvNJnb8nwAEdQDFriBVz877wTdls+8Fmt3WbfJPP3o71+GY+efoxsFaHt53j1r1xuNTrNx7i9Pfeb7p52l1+0sl2ftptvptpYwunnaDc69ZtA9bfhe4Hbap60WYHO+FECv++PRJzi6CBnBuqzb9mDaVod1Wv55x+ss2+dBt+vWu8vG6el5vdnonHp+y28Fy0a9tWw1AGDHb/hBe+m26oF+dFNvlgh33mk3T1unrN72l9163TurnwHkZrfTcFmDBWfdM0S70dWPPt6USb50z7yge948bzU898x126ztNZd1r+nVXXbeabElawKGbT6yuhLfcxtd1q77jJ0G9a57enbKWDtoua2zRh2o1j1nrTN32fFaHhDx3AeEOg3v3Ot0g06wDHz9aPB5OPjlZjIa5zAZ68Ba6h2v6TfPALzHmnXWbJ92O+5pEHRZC6cEfJtBq33e7Jx77WW90+oul922u6wjzNth/+K3HJzXPQ/qbgfWctYAvmJBs35+1g4ay3r9vMPO3a53fnZ6HrhdxlwXCNGG2dzuWb3rn0NH2MPBZPxpdAmwvulPse96wOpfmd7TmvVW19T0tfvspOx3Z8UibDxrQ1vksLUXiMcjLf8PXqzcF5bAmxZ1e2BuIB/iZO1kLxsErVMvbNG/H82GfaDSrfNpdDWcEhr3DYfLZSYOlvP7HyzCswUny9q86OqccuqEpdtVlp6s3ShcsjRzEtAA4SazUDDqZtGB/xXNrwDi+s9ZMhdFd2pFm68qmAzkNcvKr7+T3BrMhhfO9eRieMWXwwHpPd1vdM/PAwZs49ZbQaPZqNfh+HfqQbfR6p75wbJzxuAguPvQAsT4jAAHRjRbLd/vtn3fh1PlL4EJW3ASTutt5i/P612v4QZnjIh7fXPVnw2dj/3p8Go0HgJGdat+1qw3gRXg9J2dN8+6jSMQpKOxMzZR/eGf6XCIog85oAE7Dt3bjUajeTS8mQw+T03tY38GmzZ1QEo71GZqdzcXMBEuudE0NRjUaR596V9dOYP+DTXW60dHRwFbgm5mgQH6NAhRjpqgWgNW69GawyWpi/xlL6dE4oYp0263EaqkYZLEiZHrhelscuPo2jsOScySPrgO6isDtaAA/wCICL1qwXtgX6NGL8guQf3Ie1vxhkWGnnh6DRXcskADrQlQqJEWZiwxVi6cA7enLa0EON1oaR8+AK1qpubpeq1X2sUHa7sJwMoxPD5jwoBvImh+YM9BeA8ca6iYey8ZS6Fvr9S5jLpX2zP4PswcbxV76rI9WLayOMRVTFA7AL5heDpC0ZCqaQYrBXp4tZoF9hjQ2KhBs6f/bx1f70WD2whwjsEooGmBJgAPrAF3LTcbm0uYUStxBxlqYco3EPYgCogvZHP6sl6F0aMBlNaFrrweTaej8aUzuXWmv10Dq//i6O9osgJqmTaabQuc9Iqydgaf++NLOMUqhHTDfMC2bIdZ2OqgsUWoOmCpklVn4DhTK1YVe//aHQzGzxYG0XAEZOA/YraXVLxO5whrAaMBRo6JhTYmSyz2DPPzjga8L+0n9ue7gVb7V+YIM8sAS87heyKEnEBEYRmwXSdkBbtLWAeIY+ZnLAAzFwAEGjCya2m3zNuGqwDs93i9yeCYoM1JOy7gvodN0/4OspubxxYaxPlm5OdTYkMbUjFTYG/UFp2vD9DNiSlYbHchIMTKuwqwAMOvoElInQjVIqRzbQ9ie4iDKJaMM4lg0ShAgW0YgmwDpQWoAgiLaOUk8R/y1NFkeK6oa822hRAmbsd2GAutQiLrYLte38ycweRuPJvqFRDfkrl+n8Tbjb4gEZWgiCK43wFEo5nD3NsRJsJuqKYvbyd3N3wSdQ48fK9N8j+H4eZAr4b9XwRMpAKQhca+g167i+E85XBJp8LEoYhsu1XQ5OLuRnKG0P6C5vJRUBy40N+uQAgHTm4mwAzQuRC5aBdYAbg2qSH7mCny2SN7Se1ZsmVmysBRQ0cwtQ3dhP/1dEUyKlQ7OJ/gc9XehNXkj7dDMMqms9s7ckx0RUUJH8z6Z7j5VD47oKa+FhqHpv9qwYIy6AISFNzdcRzBUdbxKA1uBwJqwfUsgwEoalYhEgz5UbXMxEhQ6JMvV6PprDoeOcSNXowQ5XMQJjAn7lmIe/bVCqNlLOAKQBcjWOVscvtbFdBXrkvftgLpMJLy2HNQS0qukPAjEiXKlGuW4f7TrpPbns//ukFZK0Dwjq8DUY1OZShS6BEpZOgqgyDLA85iWchWsrViVHC5rO1hJ7SkHq0UbMYsRc4pJgC6sRWYUiUxVoJK24CEscDeNB6BE8Q8KETESspvANkB8goozMfKdir9dXL2gY5PIYUGdBhfMZhNnf7CCfh1NAXen+qvgFOCN5w3gMtYghEjB9XiFs6dTpyPpxaRL5vVKqjLhiMjF46IOlVgfHKRaG8BEaEsB0NZDhqRpeFAJnmmpoPJzfC1xZFsJHWBZGp1q8TX8SDI12ij56BLGiKXHNGGWNMIY+sjnorRZIdJ97k0cF5Bdcd/OJvQf1wxm9ZB0ibo7TJNMN/h44UFlvLaACR3RV6O8vjmn/kbBXEydnyQAKgsYAFukrgvOAloB4/pCzNAh9KGN0tYWwYqwY83LySka3uRewiDAPzYBZq4MNA2YAj6uTVApX97OZw5n0cXF6jOP/dL21OIuQ3It2UYwaJUgDULqGTsgPk0Go9mw39jK/Y5n//NrdhlbPzvAMUa9WZLkam4eUQvtL0P7OIP0U6BVyIc10/EB6WNE36BApV3KjajOGy3Q4FjCbIwlPEYmYLThMXMjx+GKFKjOIqmaBfz9PbYcdSvYsypg/IVSJe7iYjy35Pb62lJMqyZG0k7SY7HNqMuvIQsqL6GJvmWR1jp5HxlSZwaBhJt59DwzrmyTtzonhktRdPAsavMMp8/ct1lkk3Gou2aJejcKiRADQSGHHDcBk0gfWHb4WJRFngAOadHo8lJISMWnCaqJZqvaR6iO4SDVWIAXmGwdVcVXLVjJABoQv/RmPPhJbSqtixhv1BAOpzKErxKYcnWyG8qHyJatZw9q28JCf66OCSiHwD/ucGO26XBORr8lYl278ch8YtzMbwcjoe3/ZynwfbiNoGhUuGYUDoBGNwSjqMsiVcrpq7sJJ/naP+ZJcjidO2cvQJkfv4UNMcYZdw9ed90xEvv4b9o8WSB3oN/4KfcIIeopffoTzkwpuf46r38Jwz9qPc4qpabIq8bktdPgcHKEAZ6r8B7T/fvUhxIbNYuK8kDqYXLkoG3l6VEzpZvCBUeGuHHVnBtNK+SYzF/3uXhZ+FzlTmY6EzrIU71UmNzrGDHj1BNxm1QMTlZ/MiiEGSGQUGNUqiAPyQuqCHu9qtSMHed83EgCauRdVP/SHL4SnWXvQ074NLzCdXAfuHIY2YGRkppWbbByTUoEOEeQcaewXbaZsvjrq66amtuVIk5H8MoQLtKvxmNx2Cczkbj36bgrIyGUy6XdL4ILqMm4yvpxXAnEMUloGVBF7QuLArw5L7oWkhnJHJqzxe5+EXcuATOiVsI4TBAI0jAl84mjqjYkM0PNuoi6F77YGOiB88pnsf6B/v5A6JDLc9/oZc522B/U/rTs8kvYB9cDceXs89VG1ViEDDCAMfZNuKBBw9IAq7++GJ2W3Ky+FItdwMGcUBDVN7kb01tvcdiEVzpPzD/EbYRjPvMoAfKJKKfWcSrBnk7xafQ6s6SbYphK0roussMdobHp4DEx+gcajxsSSShvAqfKo9UyTRonPgPYp9WsHqbeKuCCDBYGq+ewArJktDPFDOUCMdHFt4xTina0OsGZgNUN06j2a0Dd930b4fj2Y6X7G9sPuZEL2a3Npn+XkP2fsnf0mK4v5nz1xI3eu5vTOq6qEQLlnvirctysFViRWd39tvNcG/YzN8AR1QSXwgw70DTQ588kZUD/tyfSoZb28pRphE/doCBLGuXzi4IBytrWXzsCmS79dTQCRPZF+hN3hOQvKyMRYeCxASwTPESJOGd71v3DlQKSmBfnwR8ZtQs5HiQtxjR4oQg0ujSyAMZbxMLcr8BdnDtbvJYMyC22ermHyy8f8hSB3m9ynsE4hCBhL9rKcsThCpQVwFUqKa+29SbjgyyY59SIvcQQKqsuCfi8VQkAg3TMII+gJeKPAUQdDBh4XwV7HjT/+1q0r+oxkRp3FyMWdR+btTzEV+Go8vPs6kDtubst7JFUh4l5M/afUQtAv4KCMBnF9Qu6ngQZA4GtUjbh36ew6rKDEDmPnsAUS8EMw0CG1aJkEmgi2IJKLNBmEcfckEdYU8BjFZ/UZbRV/bafTZkB2p6Fnwj7H9EQE5VM6+kLyB5K7o3+UpssSCC8SLeL7eA0S6I40a9/kNwKHUWmo9ljyGHVVJ1dk6mx8JleJ7D8J7UbgsxnZuCTIjSOMHWH0KEFlWCddzYC23e6C1+CCJtGBwcw3j5i40EgSO9BU9bONzA2vgYHTcO7mP/4/BqWubEZ/PFrCKVB0Hk8ONGGZ8ddKVhBxP/4SaBgxEIB6Whw/1vgzjdzLUwvdrEKaVf00MsnbpPLJC2CxnIeEIKiFuw1bGiCQDykIayvTRWGgPiLc/OxvGjTehYS4rErSKQ3vfABSxxJP7YyVBmk9mWl2KCVQwyKDUdDsvgKxM8iHBXSl8EB5OsY9DbVfmBeKKg41IOn+b1hRUF4dq2T03982Tyi5PHLyfjgVSHHDFbjpiLTeSOtbCFyOuFl3L3RLNgKrO8CwuVLcTq+CQyY4xZOUwBKv43XyPlZFKbYu8oNckWxVSGcTCZw4dXiMHhkNiX4QrK6kieXQEdtxv7231vHnKx9nZoQExv2/cchXscwCf6XpaDiAHmwlrU74nOD01oPbmrLYb4ayWsnOnon3I38AiWzGy2if2HItLBSxpq5RT/xg2TostOtUMlDO/aHOt58ycc2ON/3jUXpV6EiOR7jv7cBRZYvMsfGotFhe40CFYuqyukNY/vXGjvErquqEhYozyZAq4Xd2ChlZRDyoAlQXx4IRjw24icQdC8YEKwCLdHjF6HERctMoBVCXSg8HHY71t3ZSBEE31l1AsymtVo1lQURtcf+1d9OBoXZdlGswnmhS0MsTLCkUWcitOm+JZeHKtvTJ0X+I2HX0Tdniz1A+Pp73dgN8spN3bFE1TKDPbVGFRs3nIdIdm+ArCrmqibg+ZpAuQKExCX3zBjwZJwDV6MEwZ6r6hyNHXwpygrovfKE1ZrgfSIYQR4FWdk6ek9SklitixPYKjJhx5PPFSBCD8X04fCdu1VM+/VIdz7chSnR44s27ymSPPQeQ+jeweWvA4zTKRwXKuAscKM18akDhYoOG6y1nudJhZbbaMIZL+cKPcgaEMdXnLh1Grf87P9aJJwkCQn/QvSoeLqFImtp539zTNYSk+dSo0Ff6pJpZ3RsKNXo8GolDsvw8gwJ0t2u17Ls/JYYePO93RZ1Go/2918mhlubR+TwrLeRUz0N17UTNXPm4Q9wU6lmuv7IeiIzF1pPJeF1djaKKP6aEzXan7yssni+8TdPIS+lob3EeUbLPWoulLHwFocCuwdsAl4SbGF/QysH6u9xxDWnlauD8HX32K2LX8hoiL4zt8GrtqB5q122qbMCViG3BWBegt96HUPhyB7gC2n3QGDIeL+EM+ViJVg9h7rdUC0J5sEzTYy+mWNmL23QowMNyzqwSpzYDGsCcg9K8FmCp/l5V44RoZrynKh1A2ggvLPgP3BagR/Dv+Noyy838ZbgGxRsTX8zWKR036ziox8Fm4BgVuGtUh4fqWvgtbZ+qB1l2+1LLdhKxuAWJc3MwRmgV83IK/N+Okn7rTVEDVjxyZGs3eDyLOVKAjYIPG4kUf7gBuYkol82mmdn3baJpX18TQwuGv961IOgI+kKCWR3kGjxhBr2wm8LDf27mZzEMpoQUssVhWGcxRZV2HE3MRASdjcWVu8ycI1BkhFf3q2+oG7/mKQqbVngdq7fVWdpf+M+QJNI9gZ8M77Os/TEzzErQyuZr4JbpXYDXbcMj3gq9Q2rHPTOj+riUABHB3ffbGteqPESdwbwOnMfJHmclPKigF+ZZaCBvKGTU2NalOcu+A3qkVwsHoTuoX+ozRpDjEhVZPJ7ZCvi+J7zHt+KlgiR9lUttqunAGBr8AID4AIsDAwkffZ0MK0TVOWW5A8SW7T4kiw52D4zv0Fd66ib/gYKXaxDoenBal5b0Kwf3vtiFTyToyZsOCVfDtOKwdZcgJFKuOQR46LN0NwanyWlg12Isu8h3u1MBtVHYobaa1jWFAchb5ROy72V/ugycJfU7+8uXNmYOM4H+8uYDlq6Q2dEaK2AVqBWDx/gp2koIUDyomKk5wsdiKgXCW7j04y+Em2GqERqzEpHlDx+6X3ZL/hDz8j1GIY2ah2xbtXgKaps1rbnyw/wZ8MM0ogrvlkfAdAdi9XqNojow7UNF/yx5oJehfLnABv9kwTmGC7bH0e3IOp9NoJIVCa0BazFmGTffyXo+dunwG/dYo1qmlKAsV4qEncTMFQc0G7RW0HAs33rm41fpLwygxB9jlnuTwniD3zIEiR5h7v1EVgR8tz/UfcE0PNHAhxjPWyqeWvwGBFjqAkmnNQ1r4tav8dgduw6uYPAmZYoO6ES2RZTosK3yJ/4wFT1gqIcG616PTaOlpB+p7SjpKN9Oys2RoroBKWsuQJKz84y58YWKfx00+ntQ/os+M5/BW16dXoejSrkJ7lfinnB9qOsjlSO6QmFHlrclhCWTDgWdIVSmfUFOJYci2BuS54MLmyeD2O+uc0goXzgyihneS/eabxxSFv3f72/b0fJ5gnBtn+Pt5mm22Wi3kq1BHcnFeVranAsxwpCE1YR1lwysVVogT7RNVcHIl34WKvuPoPRBaxC0cpTmy+r/89AVWelyIM9p6AaLmb3ArwDjLgGHe7ygwgXB4ZMud1OHL1xf5Rc7XrYl5fvLPz9f7IgAYMIETLUZpDUpTUgF2Rm8JUF9Y54LAzJBcvH/UeumZocSLZiOHnvPxh8U48YSnD4qenHSAol/aN3qkKoBUqdQEScFEpsQ+84H0pA4p5pLuVYTAWdQBu4hMss4jD5XuYR+LUMa4HqItRjbdH5WEv+Z5ifmQrKfB+rps6VsXzAKRzPZzdjgZ5fAnjyfY3Ko+MViu9V2B/ogIxqQf/nXfib8olItiNcOHgvj32kAAn+XpEyCFfkfAFv39/wyjEZQrC51X+tCxeNXNzO7zYqX7iq5vz2k9eJeBRTYq+sEslHvgn5ZkCOcexELVFxYdaAUeAZUCOJeQXgeoDf0qKMdf0TN+WjfPHRVG9rPc/DpSIB4W6JP3NCgErkV3eHyyGdOthCXoBvlaFb+q/Dm8vRoOZ2PFKCOSeXBMb4yhy7sWxrzzUTtRXxRjPNrzDY7zqGM9Nmb0GK9Xbuw/mzpU7eTOBOTgdTAYA5CSVwTAdvFQD93h0RN+Chov5/eKDu6+1erTzsPpu7yK5vMV7DBi5mtz2sdZAvRJ9SUV4/fHFCJ01vSoqOdF/RjeSjiknaPEsV630oHX9bJ8eVQWcVsJhPHEGV8P+LWFQLRorgqUcfxMEHEi88Ik5YrOcgfOUOnDoOIY7hWP7en8Uvb1DvfOd4v1TUKIRxlwcUUOm9+R6KxDwWPOFw0hwxzGhlT24ESFI7ZUBuWDH6JXCIb0d7qoMjGLMlK0xbgZKO1yGPubQZZQ4DxFfNvIyd/y4gt7TP0+uLnRZ7gYqdB36DlZTO/IDEKIeCx36NAMgpB965StCyiuLPYMdnVIE/W52cyeqRob/GE1nU9zdCZzmL7eFG8A/DYH3LsVE+fXL1KRmBbipP4t3Sukz/4iBhUhjHDT2/mWAQWWKdEbjw4dmvZCkAWjwbQrW7fsY7J/0JfINaIJhUVxSRpXLYJjJlS0qHfJFDiY3vyllbbKoYN8YNf3hcCntYLQw3bg+AzsolklTkjgU8tZPkNPAejsJEmDIk+uXC/qLH9LIsvVxGj4fFwbxMaUJjp8aelGbJO9fxRmHSNNU3vK7CPDC4gET20YM+AUEbOXlSvz7CKYmVz6ezJy78UUpJTPuXw+nN/3BsHqXjODkHKL9IIvk5dqxVFV/JKBuHa69wKb2H1HdOdjBFJrE5NkICvelJt5DKirGvuBoTbmbw9NryVorshwY1HpgkVYUVFGK/D2sBT9qEsM/hMThkjEi9oFdltgW22KtH7E+jFNeXDgjQjnxowxjk8R8wti+z0DVipKeHhX0AAtY9KEKi77Sgg5Yfo9KrBQYYicJ80pmSt5fR9qZuiArluxy+lYhFbK5akf86SwVlne+1p9yYXmeKb+vsz/NpH7ARQo/AYjbJJ6t3P2T5MXQGB6BZvUSIL+KEbmR2JT8/t87T/9fUXQnjFHc1hNdvQJWCDzRyu+bP++5b24Rdxle7f1yV2QtKyILMxcsBZZQJ+sphXdUbFGky5DVpX1VzBmkAueljtzzDf757qjlZVj+Vr7SAkMOr6EI5GBJhKEeyDnAXpjLsoP3AwstFjvHqemGtCKZU3mgqCaS079aESnIj8WBAnPdpD0EXrJlpqaykpxBjD916nCOPYfuzZOHi0nz7Cdf734gxaGTjDsv2hbqb7xhAQdpm+bH6Lu5PMTltR0+RN7+IVYUslqiUyrkDSP0SAysD1ItCFJI2GjRbb6v4aa2Jz9/oJeSsc8z2UqunoJNvOTdrtyGL8Eyiyeh8Tl+6Ebbav0+Apz3RPhmIUAXDdLot0s3ByqD6FE89XZhyIa8h5pU9vB+8vGus/GhwY67yjUa+cK5uB19kr4khpqy+BFZy65U/NP688e8ap3TxdswQRaTX6J5zqp3s3cKWKhAvN0y9ens7gKLL3l1eemWonoPS9yg4AFBBVGZvE3YcoWiyODlZfebrZDAPZEJ769W1SpvYrsUx/rgbmr8y2F5Tpxr/ehFxEGgN1b9pFaVcXZxsnf5eb8NUP78BNESa2Vzo1ItlDV19UkWsXBLrFQIz8HQ7zwDBeSoJmqKaDEcE/fJDVcYLBInSp+1dKr8KnrhHVAehyRDxajDkfp0Oxzih7EqpTWFmfwCmACbo+9RunUMZ3L7LCodyiasXstr4U19MLnqf6xC/xGuEDk9wRzJNlKlys427JYaEQ3V8g53VXt934tJ7QovIisWUhP21y7dZ6nsulnec+VGC/QRFACGPGQ5csSpVZaAjGNxSeeEf5NOflUNP74hxGfp23s8W4uppROsONHQfsaiEZYK3s9rE5E3+IPBsxJ58CDJ7Gom8L3CS5iUAMOcuY8yTYF6X34BA/ByhHGI0XheVAo/MGESuisbM6XvudClcLfIxH77LhY8iLF0GM53/+pKk3YgGjWp5jFgRUb6HduwIkbDvCt44vyiJy8FxPCj+EqhUkbjPsV43Ya+YQjulZZtCTB6QXiXhD4c6CarkJEFpacEVZhcb9hWauJkI/IldjmtLrLUMlVS0RlqosSkDShlkATp5PceepKUy03ltsfGtsU7nm3GjNyof5WXxqM7B8exf6VmnatbQeabLX4fNxblGl/+9puopafI6U6NywqQFJUhIpC6W56Rh1T3Z990d/ssvjRm/OAclBPaO4UasOVRKSRj7dDMWKmGGyWr4+hyRA+vRphYvEYWHXeU3+mO/g66WuA2gJoVRjg/ZSuFLSqpTzpFYEJkLw59e0epRnqFwf50sc7h0ppiJ/NLEbt1Nn+yfKYaGKfteG3ifIcPzK0IEs50r+Ue91tdRUMpFXlwirn4QgMlw6msXF/Y+w/IKzBE3SMMFdUw/xFX/A1ULLIhnn6UfPxDpkHo3kcxFsiZUuCVr8SRekChiZnq9MH6D4q+/lx51p9gGZV0xeW4/uV4Mp2NBvr+HQfe+jd3fA/1X6U817sg694uj9V06Wv8WOkt7y4/nQJCJY9+oDVOJbASzs5N2zIg7A+IsYS+ciLGCIumfHWrV7JNylCE1hAYybCABCd1SnkMW7kb0J3OH7iNKcPP+2HSbW8JUWUoEpwbELLAAdD0EO9XiyCU8oc9+byEof/rEPvjN1vihH9Ahn/zdydaTnEoWQ1HwX6wagv/WYkWExvM96938aGoihrcXd+Bezb6dejIxtJdSgHnwOIXVNGhwKDCjgJGHhvcGx41lbNkkpQ1VVeXF0kqobBvOtYc3031nihDHv5jOLjDOy8Y6FVzNli4Cf7EoSIZ/XY4vbuaYdImQUTKMbGDBY36xXBAHwPaH9zA2waj4Rg/9NZwJrejy9EYjBaZ0MgjG7WiLhYrzEROc2PLzy5b/eR+i1GYG3xKMFLvJ+GGsu2OE8S+4whJtLHcAHhHdDf042M8CrrpP8RYOmXP9d9dHQ6UdA50MgUA0fwiQCEB9wBDRjwG6aC/3qP03bo9neCwv9knd/OPi49DHeoL8uFNeFzOH35Pftax9LNe6UdujXwPjo4NnWiXsJv0HsiCuKfPJGK11O9qrRRnYv3vfSokdqhofkBFlM7dFJ6HWFk8Gzp3YPVSPOzT5NYZYIlU/wozncVn1wpHtFQGzj/aJSU5bLESMtK5aM4fdqMpOpe3eQ9JuIrrDp64m2U8JmPSlyjVGlJTl9+8VD9TWaFKwYXKp1QPubav+LVK5uEtIXFzOxxc9UfXdImZSD75BdYo8p9kIqGJnLeAQNZ7rdcq63TEWI5sdU3+TOMabfMHdO5rsN/Uxsr+/Zii3fmgMKgQoKEaczXxY2FSbl1NBr/sFVhv8l85cAFElifn/4GR8lgKyE1gKodiD44DPOU4KEMdR7AUF6hH/wdKCUZSVWAAAA==')
source = gzip.decompress(base64.b64decode(_COMPRESSED_B64))
assert hashlib.sha256(source).hexdigest() == RUNNER_SHA256, 'RUNNER_SOURCE_SHA_MISMATCH'
work = pathlib.Path('/content/ttm_six_functional_transfer_v1')
work.mkdir(parents=True,exist_ok=True)
runner = work/'ttm_six_ttm4m_three_arm_v1.py'
if runner.exists():
    assert hashlib.sha256(runner.read_bytes()).hexdigest() == RUNNER_SHA256, 'RUNNER_EXISTING_MISMATCH'
else:
    runner.write_bytes(source)
print('RUNNER_SHA256_PIN=PASS', RUNNER_SHA256)
# Torch is provided by Colab; do not replace its installed CUDA wheel automatically.
import regex, numpy, torch
print('ENVIRONMENT',sys.version.split()[0], torch.__version__,torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU_ONLY')

In [ ]:
# 2. Download exact P02 architecture and TRAIN-only BPE implementation (no weights/data).
import urllib.request
PIN = 'f69d4eb0b566344b19e9795de59ee6eec87ef47f'
BASE = f'https://raw.githubusercontent.com/minhtri22/TokenToModel-Lab/{PIN}/'
files = {
  'p02_core.py': ('src/p02/transformer_core.py','9762343e06cf800b50574c2871ae1ed58537fb18'),
  'ttm_gpt2_bpe2048.py': ('tools/pilot_t1t3/source_t2/ttm_gpt2_bpe2048.py','fa5bd892941ba5aa6e6b2f0b2b0ae974efe297b6'),
}
def gitblob_sha1(blob):
    return hashlib.sha1(b'blob '+str(len(blob)).encode()+b'\0'+blob).hexdigest()
for filename,(path,pinned_blob) in files.items():
    dst=work/filename
    if not dst.exists(): dst.write_bytes(urllib.request.urlopen(BASE+path,timeout=35).read())
    assert gitblob_sha1(dst.read_bytes())==pinned_blob, 'PINNED_SOURCE_MISMATCH: '+filename
    print('SOURCE_PIN_PASS',filename)


In [ ]:
# 3. Prepare inputs. Drive mount is used ONLY to read the existing STEP1280 checkpoint.
# It does not mount or access the original G1 private TEST (which is not an input).
from google.colab import drive, files as colab_files
drive.mount('/content/drive')
# Upload the Qwen result ZIP and the exact frozen TTM tokenizer manifest from your machine.
# The upload UI is intentional, to avoid enumerating other Drive files.
print('Upload 2 files now: TTM_SIX_G1_EXPLORE_REAL_QWEN_DEV_RESULTS.zip and T1_BPE2048_REAL_TRAIN_MANIFEST.json')
UPLOADED=colab_files.upload()
QWEN_ZIP=pathlib.Path('/content/TTM_SIX_G1_EXPLORE_REAL_QWEN_DEV_RESULTS.zip')
TOKENIZER=pathlib.Path('/content/T1_BPE2048_REAL_TRAIN_MANIFEST.json')
assert QWEN_ZIP.exists() and TOKENIZER.exists(), 'REQUIRED_UPLOADS_MISSING'
# Exact Drive checkpoint parent. Never scan unrelated folders.
PARENT=pathlib.Path('/content/drive/MyDrive/TTM/ttm-small-pilot/ttm_small4m_step1024_to1280_exploratory_v1/checkpoints/step_1280')
print('INPUTS_SELECTED',QWEN_ZIP.name,TOKENIZER.name,PARENT.name)


In [ ]:
# 4. PRECLAIM: read-only SHA/READY/tokenizer/prompt/tensor verification; NO STUDENT TRAINING.
args=[sys.executable,str(runner),'--mode','preflight',
      '--qwen-zip',str(QWEN_ZIP),
      '--qwen-source',str(work/'g1_explore_public_qwen_dev_v1.py'),
      '--bpe-source',str(work/'ttm_gpt2_bpe2048.py'),
      '--tokenizer-manifest',str(TOKENIZER),
      '--p02-source',str(work/'p02_core.py'),
      '--parent',str(PARENT)]
# Retrieve authored public Qwen source from the user-supplied cached ZIP, not new teacher calls.
import zipfile
with zipfile.ZipFile(QWEN_ZIP) as z:
    b=z.read('g1_explore_public_qwen_dev_v1.py')
    target=work/'g1_explore_public_qwen_dev_v1.py'
    if target.exists(): assert target.read_bytes()==b
    else: target.write_bytes(b)
assert hashlib.sha256(target.read_bytes()).hexdigest()=='9688deb1d1157c93fb7090e5b1c12ea4bc56ed0dedadb0597d84ceb4e68e82eb'
subprocess.run(args,check=True)
print('PRECLAIM_SOURCE_PINNED_PASS; NO_TRAINING; G1_ORIGINAL_SCIENCE_LOCK=HOLD')


## 5. TRAINING GATE — remains OFF until a separate owner pilot approval

Current status: `OWNER_PILOT_EXECUTION_PERMIT=NOT_ISSUED`. Do **not** create an approval file just to make the notebook run. Once the owner explicitly approves the new pilot in a later turn, place that approval receipt JSON at the input path below. The old G1 science approval and any previous T4 lock are *not* accepted. A failed preclaim or missing approved inputs means `TECHNICAL_STOP`, not a request to ignore SHA checks.


In [ ]:
# 5. TRAIN — DELIBERATELY DISABLED BY DEFAULT. Never set True absent NEW owner approval.
AUTHORIZED_NEW_PILOT = False
APPROVAL_RECEIPT = pathlib.Path('/content/TTM_SIX_TTM4M_NEW_OWNER_PILOT_APPROVAL.json')
OUTPUT = pathlib.Path('/content/drive/MyDrive/TTM/ttm-six-functional-pilot-v1/ttm_six_ttm4m_functional_transfer_public48_dev16_v1')
if not AUTHORIZED_NEW_PILOT:
    print('PILOT_TRAINING=NOT_EXECUTED; G1_ORIGINAL_TEST=SEALED; SCIENCE_LOCK=HOLD')
else:
    if not APPROVAL_RECEIPT.is_file():
        raise PermissionError('MISSING_NEW_OWNER_PILOT_APPROVAL_JSON')
    new_args=[sys.executable,str(runner),'--mode','run','--qwen-zip',str(QWEN_ZIP),
      '--qwen-source',str(work/'g1_explore_public_qwen_dev_v1.py'),
      '--bpe-source',str(work/'ttm_gpt2_bpe2048.py'),
      '--tokenizer-manifest',str(TOKENIZER),
      '--p02-source',str(work/'p02_core.py'), '--parent',str(PARENT),
      '--owner-approval',str(APPROVAL_RECEIPT),'--output',str(OUTPUT)]
    subprocess.run(new_args,check=True)
